In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
REPO_PATH = Path("/content/IT3091---Machine-Learning-Assignment")
if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
os.chdir(REPO_PATH)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
print("Repository is ready:", REPO_PATH)


Repository is ready: /content/IT3091---Machine-Learning-Assignment


# Notebook 07 — National historical-yield reference



In [2]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

PREPROCESSED_DIR = REPO_PATH / "Reconstruction Outputs" / "Wijesinghe.D.T.D-Crop Yield Preprocessing"
OUTPUT_DIR = REPO_PATH / "Reconstruction Outputs" / "Wijesinghe.D.T.D-Crop Yield Outputs"
yield_path = PREPROCESSED_DIR / "wijesinghe_yield_preprocessed_15000plus.csv"
model_path = OUTPUT_DIR / "wijesinghe_yield_model.joblib"
metrics_path = OUTPUT_DIR / "wijesinghe_yield_model_metrics.csv"
for required_path in (yield_path, model_path, metrics_path):
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Missing {required_path.relative_to(REPO_PATH)}. Run Notebook 01 through its final push first."
        )

yield_data = pd.read_csv(yield_path)
yield_data["Year"] = pd.to_numeric(yield_data["Year"], errors="coerce")
yield_data["Yield_t_ha"] = pd.to_numeric(yield_data["Yield_t_ha"], errors="coerce")
yield_data["Extent"] = pd.to_numeric(yield_data["Extent"], errors="coerce")
yield_data = yield_data.dropna(subset=["District", "Season", "CropCategory", "Crop", "Year", "Extent", "Yield_t_ha"]).copy()
yield_data["Year"] = yield_data["Year"].astype(int)

model = joblib.load(model_path)  # Load the existing Notebook 01 model; do not retrain or replace it.
existing_metrics = pd.read_csv(metrics_path)
print("Existing model metrics (unchanged):")
display(existing_metrics)


Existing model metrics (unchanged):


,Records,TrainRecords,TestRecords,MAE_t_per_ha,RMSE_t_per_ha,R2
0,60529,48423,12106,1.993559,3.096213,0.675609


In [3]:
# The reference average uses only National Total records strictly earlier than the forecast year.
# Edit CROP, SEASON, DISTRICT, FORECAST_YEAR, or PLANNED_EXTENT_HA to try another planning case.
national = yield_data.loc[yield_data["District"].astype(str).str.strip().eq("National Total")].copy()
if national.empty:
    raise ValueError("Notebook 01's prepared file has no National Total observations.")
national = national.loc[national["Season"].astype(str).isin(["Maha", "Yala"])].copy()
if national.empty:
    raise ValueError("No Maha/Yala National Total observations were found in Notebook 01's prepared file.")

latest = national.sort_values("Year").iloc[-1]
CROP = str(latest["Crop"])
SEASON = str(latest["Season"])
CROP_CATEGORY = str(latest["CropCategory"])
DISTRICT = "National Total"
FORECAST_YEAR = int(national["Year"].max()) + 1
matching_extent = yield_data.loc[
    yield_data["Crop"].astype(str).eq(CROP)
    & yield_data["Season"].astype(str).eq(SEASON)
    & yield_data["CropCategory"].astype(str).eq(CROP_CATEGORY),
    "Extent",
]
PLANNED_EXTENT_HA = float(matching_extent.median())

prior_national = national.loc[
    national["Crop"].astype(str).eq(CROP)
    & national["Season"].astype(str).eq(SEASON)
    & national["CropCategory"].astype(str).eq(CROP_CATEGORY)
    & national["Year"].lt(FORECAST_YEAR)
].copy()
if prior_national.empty:
    raise ValueError("No earlier National Total yield records match this crop/category/season.")

model_features = ["District", "Season", "CropCategory", "Crop", "Year", "Extent"]
model_input = pd.DataFrame([{
    "District": DISTRICT,
    "Season": SEASON,
    "CropCategory": CROP_CATEGORY,
    "Crop": CROP,
    "Year": FORECAST_YEAR,
    "Extent": PLANNED_EXTENT_HA,
}])[model_features]
model_prediction = float(model.predict(model_input)[0])
historical_average = float(prior_national["Yield_t_ha"].mean())

comparison = pd.DataFrame([{
    "Crop": CROP,
    "CropCategory": CROP_CATEGORY,
    "Season": SEASON,
    "DistrictForExistingModel": DISTRICT,
    "ForecastYear": FORECAST_YEAR,
    "PlannedExtent_ha": PLANNED_EXTENT_HA,
    "ExistingModel_PredictedYield_t_ha": model_prediction,
    "NationalHistoricalAverageYield_t_ha": historical_average,
    "HistoricalYearsUsed": int(prior_national["Year"].nunique()),
    "HistoryFirstYear": int(prior_national["Year"].min()),
    "HistoryLastYear": int(prior_national["Year"].max()),
    "HistoricalReferenceType": "Arithmetic mean of prior National Total records; not an ML prediction",
}])

print("Existing model prediction beside a separate historical reference (both t/ha):")
display(comparison.round(3))
print("The historical average is a reference only. It does not alter the model, its features, or its R².")


Existing model prediction beside a separate historical reference (both t/ha):


,Crop,CropCategory,Season,DistrictForExistingModel,ForecastYear,PlannedExtent_ha,ExistingModel_PredictedYield_t_ha,NationalHistoricalAverageYield_t_ha,HistoricalYearsUsed,HistoryFirstYear,HistoryLastYear,HistoricalReferenceType
0,Winged Bean,Low Country Vegetable,Yala,National Total,2026,34.9,10.226,9.377,12,2014,2025,Arithmetic mean of prior National Total record...


The historical average is a reference only. It does not alter the model, its features, or its R².


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_path = OUTPUT_DIR / "national_historical_yield_reference_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved comparison:", comparison_path.relative_to(REPO_PATH))


Saved comparison: Reconstruction Outputs/Wijesinghe.D.T.D-Crop Yield Outputs/national_historical_yield_reference_comparison.csv


In [ ]:
from getpass import getpass
from google.colab import _message
from pathlib import Path
from urllib.request import Request, urlopen
import json
import os
import subprocess
import tempfile

NOTEBOOK_NAME = "07_Wijesinghe_DTD_National_Historical_Yield_Reference.ipynb"
COMPARISON_FILE = (
    "Reconstruction Outputs/Wijesinghe.D.T.D-Crop Yield Outputs/"
    "national_historical_yield_reference_comparison.csv"
)

# Save this open Colab notebook into the cloned repository first.
current = _message.blocking_request("get_ipynb", request="", timeout_sec=15)
if not isinstance(current.get("ipynb"), dict):
    raise RuntimeError("Colab could not retrieve this notebook. Save it and retry.")

(REPO_PATH / NOTEBOOK_NAME).write_text(
    json.dumps(current["ipynb"], ensure_ascii=False, indent=1),
    encoding="utf-8",
)

FILES_TO_STAGE = [NOTEBOOK_NAME, COMPARISON_FILE]

missing = [name for name in FILES_TO_STAGE if not (REPO_PATH / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Run the comparison cell first. Missing:\n" + "\n".join(missing)
    )

github_username = input("GitHub username: ").strip()
commit_name = input("Git commit name: ").strip()
github_email = input("GitHub email: ").strip()
token = getpass("GitHub Personal Access Token (hidden): ").strip()

if not all([github_username, commit_name, github_email, token]):
    raise ValueError("Username, commit name, email, and token are required.")

request = Request(
    "https://api.github.com/user",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    },
)
with urlopen(request, timeout=20) as response:
    authenticated_user = json.loads(response.read().decode("utf-8")).get("login", "")

if authenticated_user.casefold() != github_username.casefold():
    raise RuntimeError(
        f"The token belongs to @{authenticated_user}, not @{github_username}."
    )

subprocess.run(
    ["git", "config", "user.name", commit_name],
    cwd=REPO_PATH,
    check=True,
)
subprocess.run(
    ["git", "config", "user.email", github_email],
    cwd=REPO_PATH,
    check=True,
)
subprocess.run(
    ["git", "add", "--", *FILES_TO_STAGE],
    cwd=REPO_PATH,
    check=True,
)

staged = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=REPO_PATH,
    text=True,
    capture_output=True,
    check=True,
).stdout.splitlines()

unexpected = sorted(set(staged) - set(FILES_TO_STAGE))
if unexpected:
    raise RuntimeError("Unexpected staged files; cancelled:\n" + "\n".join(unexpected))

if staged:
    print("Files to commit:")
    print("\n".join(staged))
    if input("Type PUSH to commit and push to main: ").strip() != "PUSH":
        raise RuntimeError("Cancelled; no push was attempted.")

    subprocess.run(
        ["git", "commit", "-m", "Add national historical-yield reference"],
        cwd=REPO_PATH,
        check=True,
    )
else:
    print("No new changes to commit; attempting to push any pending commit.")

# Close the temporary file before Git executes it.
fd, askpass_name = tempfile.mkstemp(prefix="git-askpass-")
os.close(fd)
askpass = Path(askpass_name)
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "$GITHUB_USERNAME" ;;\n'
    '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
    'esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)

env = os.environ.copy()
env.update({
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "GITHUB_USERNAME": github_username,
    "GITHUB_TOKEN": token,
})

try:
    result = subprocess.run(
        ["git", "push", "origin", "main"],
        cwd=REPO_PATH,
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout or "(no standard output)")
    print(result.stderr or "(no Git error text returned)")

    if result.returncode != 0:
        raise RuntimeError(f"Push failed with exit code {result.returncode}.")

    print(f"Push completed as @{authenticated_user}.")
finally:
    token = ""
    env.pop("GITHUB_TOKEN", None)
    askpass.unlink(missing_ok=True)